# LeetCode 4023 — Elevator Requests II bằng Python

- Link: https://leetcode.com/problems/elevator-requests-ii/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động theo đoạn (interval DP), Sắp xếp, Tổng thời gian hoàn thành (traveling repairman trên đường thẳng), Kỹ thuật "tính chi phí theo số phần tử còn lại"

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def elevatorRequests(self, n: int, start: int, requests: list[int]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Thang máy bắt đầu ở tầng `start`. Tất cả `m` yêu cầu (các tầng **khác nhau**) xuất hiện **cùng lúc** tại thời điểm 0. Mỗi giây thang di chuyển đúng 1 tầng lên hoặc xuống. Yêu cầu được phục vụ **tức thì** khi thang đến tầng đó; yêu cầu phục vụ ở thời điểm `t` bị phạt `t`. Tìm **tổng phạt nhỏ nhất**.

Khác với bài "thời điểm hoàn thành muộn nhất" (bản IV, III), ở đây ta **cộng thời điểm phục vụ của mọi yêu cầu** — đây là dạng "người sửa chữa" (*traveling repairman*) trên một đường thẳng. Với `m ≤ 1500` ta cần `O(m²)`.

Sau notebook này bạn sẽ:

1. Chứng minh tập tầng đã đi qua luôn là một **đoạn liên tục** chứa `start`, nên trạng thái chỉ là `(l, r, đang ở đầu nào)`.
2. Dùng kỹ thuật **tính phạt theo số yêu cầu chưa xong**: đi quãng `d` thì mọi yêu cầu đang chờ cùng chịu thêm `d`.
3. Viết DP đoạn `O(m²)` và bản cuộn hàng `O(m)` bộ nhớ.
4. Có trọng tài đúng: mô phỏng theo từng giây (DP theo lớp thời gian) và vét cạn hoán vị / bitmask.
5. Kiểm tra mọi cách bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Tầng `0..n-1`. Thang ở tầng `start` lúc 0; `requests` là danh sách `m` tầng **đôi một khác nhau**, tất cả được yêu cầu lúc 0.
- Mỗi giây (cho đến khi xong hết) thang di chuyển **đúng một tầng** lên hoặc xuống.
- Nếu `start` thuộc `requests` thì yêu cầu đó xong ở thời điểm 0 (phạt 0).
- Phạt của một yêu cầu = thời điểm nó được phục vụ. Trả về **tổng phạt nhỏ nhất**.

**Ví dụ 1:** `n = 6, start = 4, requests = [1,5]` → `6`. `4→5` mất 1 giây (phạt 1), `5→1` mất 4 giây (phạt 5). Tổng `1 + 5 = 6`.

**Ví dụ 2:** `n = 8, start = 3, requests = [3,7,1]` → `10`. Tầng 3 xong lúc 0, đi tới tầng 1 lúc 2 (phạt 2), rồi tầng 7 lúc 8 (phạt 8). Tổng `0 + 2 + 8 = 10`.

**Ví dụ 3:** `n = 10, start = 5, requests = [0,2,9]` → `22`. `5→2` (lúc 3), `2→0` (lúc 5), `0→9` (lúc 14): `3 + 5 + 14 = 22`.

**Ràng buộc**

- `1 <= n <= 10^9`
- `1 <= requests.length <= 1500`
- `0 <= start, requests[i] <= n - 1`
- Mọi giá trị trong `requests` **khác nhau**.

In [1]:
EXAMPLES = [
    (6, 4, [1, 5], 6),
    (8, 3, [3, 7, 1], 10),
    (10, 5, [0, 2, 9], 22),
]
for n, start, reqs, e in EXAMPLES:
    print(f"n={n}, start={start}, requests={reqs} -> kỳ vọng {e}")

n=6, start=4, requests=[1, 5] -> kỳ vọng 6
n=8, start=3, requests=[3, 7, 1] -> kỳ vọng 10
n=10, start=5, requests=[0, 2, 9] -> kỳ vọng 22


## 3. Kiến thức nền

### 3.1. Khoảng cách và thời điểm đến

Đi từ tầng `a` tới tầng `b` (không quay đầu) mất `abs(a - b)` giây. Nếu thang theo thứ tự ghé `f1, f2, f3, ...` thì thời điểm đến `f1` là `|start − f1|`, đến `f2` là thời điểm đến `f1` cộng `|f1 − f2|`, v.v. Tổng phạt = tổng các thời điểm đến.

In [2]:
def penalty_of_order(start, order):
    t, pos, total = 0, start, 0
    for f in order:
        t += abs(pos - f)
        pos = f
        total += t
    return total

print(penalty_of_order(4, [5, 1]))            # 1 + 5 = 6
print(penalty_of_order(4, [1, 5]))            # 3 + 7 = 10
print(penalty_of_order(5, [2, 0, 9]))         # 3 + 5 + 14 = 22

6
10
22


### 3.2. Sắp xếp và `set`/`sorted`

`sorted(set(requests) | {start})` cho các tầng **quan trọng** (các yêu cầu và tầng xuất phát) theo thứ tự tăng, loại trùng. `list.index(x)` hoặc `bisect` cho vị trí của `start`. Vì mọi tầng yêu cầu khác nhau, mỗi phần tử trong danh sách này ứng với đúng một yêu cầu (hoặc là `start` nếu `start` không phải yêu cầu).

In [3]:
from bisect import bisect_left

requests = [0, 2, 9]
start = 5
pts = sorted(set(requests) | {start})
s = pts.index(start)
print(pts, "chỉ số của start =", s, bisect_left(pts, start))
is_req = [1 if p in set(requests) else 0 for p in pts]
print(is_req)         # start không phải yêu cầu -> 0

[0, 2, 5, 9] chỉ số của start = 2 2
[1, 1, 0, 1]


### 3.3. Mảng cộng dồn (prefix sum) để đếm yêu cầu trong một đoạn

Với `is_req[i] ∈ {0, 1}`, mảng `pref[i] = is_req[0] + ... + is_req[i-1]` cho phép đếm số yêu cầu trong đoạn chỉ số `[l, r]` là `pref[r+1] - pref[l]` trong `O(1)`. Số yêu cầu **chưa** phục vụ khi đã đi qua đoạn `[l, r]` là `tổng − (pref[r+1] − pref[l])`.

In [4]:
pref = [0]
for v in is_req:
    pref.append(pref[-1] + v)
total = pref[-1]
print("pref =", pref, "| tổng yêu cầu =", total)
l, r = 1, 2     # đoạn đã đi qua: các tầng pts[1..2] = 2, 5
print("đã phục vụ:", pref[r + 1] - pref[l], "| chưa phục vụ:", total - (pref[r + 1] - pref[l]))

pref = [0, 1, 2, 2, 3] | tổng yêu cầu = 3
đã phục vụ: 1 | chưa phục vụ: 2


### 3.3b. Mô phỏng theo lớp thời gian (cho trọng tài)

Để có một lời giải **nguyên văn đề**, ta chạy theo từng giây: trạng thái `(tầng, mask đã phục vụ)` kèm tổng phạt tối thiểu. Mỗi giây thang đi đúng ±1 (trong `[0, n−1]`). Khi sang giây `t + 1` ở tầng `f`, mọi yêu cầu ở tầng `f` chưa phục vụ được phục vụ với phạt `t + 1`. Ta lấy `min` trên mọi lớp thời gian tới một cận đủ lớn.

In [5]:
def elevator_layers(n, start, requests):
    m = len(requests)
    full = (1 << m) - 1
    idx = {f: k for k, f in enumerate(requests)}

    def serve(t, floor, mask, cost):
        if floor in idx and not (mask >> idx[floor]) & 1:
            mask |= 1 << idx[floor]
            cost += t
        return mask, cost

    mask0, cost0 = serve(0, start, 0, 0)
    layer = {(start, mask0): cost0}
    best = cost0 if mask0 == full else float("inf")
    for t in range(1, m * n + n + 1):
        nxt = {}
        for (floor, mask), cost in layer.items():
            if mask == full:
                continue                           # xong rồi, không cần đi tiếp
            for nf in (floor - 1, floor + 1):
                if 0 <= nf < n:
                    nm, nc = serve(t, nf, mask, cost)
                    key = (nf, nm)
                    if nc < nxt.get(key, float("inf")):
                        nxt[key] = nc
        layer = nxt
        for (floor, mask), cost in layer.items():
            if mask == full:
                best = min(best, cost)
    return best

for n, start, reqs, e in EXAMPLES:
    print(elevator_layers(n, start, reqs), e)

6 6
10 10
22 22


## 4. Ý tưởng nền tảng / trực giác

**Tính tay ví dụ 3** (`start = 5`, yêu cầu `0, 2, 9`):

| Thứ tự ghé | Thời điểm từng yêu cầu | Tổng phạt |
|---|---|---|
| 2, 0, 9 | 3, 5, 14 | **22** |
| 9, 2, 0 | 4, 11, 13 | 28 |
| 2, 9, 0 | 3, 10, 19 | 32 |
| 0, 2, 9 | 5, 7, 14 | 26 |

Nhận xét: đi sang phía gần trước (hai yêu cầu `2` và `0`), rồi mới tới phía xa (tầng 9). Thang **không bao giờ cần quay lại** một tầng đã đi qua (nó đã phục vụ rồi); cũng không đứng yên (đề bắt buộc di chuyển).

**Tại sao khó?** Có `m!` thứ tự. Nhưng vì các tầng nằm trên một đường thẳng, tập tầng đã đi qua luôn là một **khoảng liên tục** — đây là chìa khoá cho DP đoạn.

## 5. Cách 1 — Vét cạn: hoán vị và DP theo lớp thời gian

**Cách 1a — Hoán vị.** Thử mọi thứ tự ghé các tầng yêu cầu, dùng `penalty_of_order`. `O(m! · m)`.

**Cách 1b — DP theo lớp thời gian** (đã có ở 3.3b): nguyên văn đề, chỉ dùng được khi `n` nhỏ.

In [6]:
from itertools import permutations

def elevator_permutations(n, start, requests):
    return min(penalty_of_order(start, order) for order in permutations(requests))

for n, start, reqs, e in EXAMPLES:
    print(elevator_permutations(n, start, reqs), e)

6 6
10 10
22 22


**Độ phức tạp.** Hoán vị: `O(m! · m)` (dùng được tới `m ≈ 8`). DP lớp thời gian: `O(T · n · 2^m)`.

**Nút thắt:** `m = 1500` không thể duyệt thứ tự. Cần khai thác cấu trúc đường thẳng.

## 6. Bước đột phá

### 6.1. Tập đã đi qua là một đoạn liên tục chứa `start`

Thang di chuyển liên tục 1 tầng/giây. Tại mọi thời điểm, các tầng nó **đã chạm** tạo thành một đoạn `[L, R]` chứa `start`. Một yêu cầu được phục vụ **lần đầu** khi thang chạm tầng đó, tức khi đoạn `[L, R]` **mở rộng** ra phủ nó. Vì vậy thứ tự phục vụ chính là thứ tự **mở rộng đoạn**: mỗi lần chọn mở sang **trái** đến yêu cầu kế tiếp, hoặc sang **phải**.

### 6.2. Tính phạt theo "số yêu cầu chưa xong"

Mỗi giây, mọi yêu cầu **chưa** phục vụ chịu thêm 1 phạt. Nên khi thang đi quãng `d` (từ đầu hiện tại sang tầng yêu cầu kế tiếp), phạt cộng thêm `d × u`, với `u` = số yêu cầu chưa phục vụ **trước** khi đi (gồm cả yêu cầu sắp tới). Tổng các `d × u` bằng đúng tổng thời điểm phục vụ. Cách này **không cần biết thời điểm tuyệt đối**, chỉ cần `u` — là một hàm của `(l, r)`.

### 6.3. Trạng thái và công thức

Đặt `p[0..K-1]` là các tầng quan trọng đã sắp (yêu cầu ∪ `{start}`), `s` là chỉ số của `start`, `is_req[i]`, `pref` như 3.3, `tot` là tổng số yêu cầu.

- `dpL[l][r]`: phạt nhỏ nhất khi đã mở tới đoạn `[l, r]` (chứa `s`) và thang đang ở `p[l]`.
- `dpR[l][r]`: như trên nhưng thang ở `p[r]`.
- `u(l, r) = tot − (pref[r+1] − pref[l])` là số yêu cầu chưa phục vụ khi đoạn là `[l, r]`.
- Khởi đầu: `dpL[s][s] = dpR[s][s] = 0`.
- Mở sang trái (từ `[l, r]` thành `[l−1, r]`):
  `dpL[l−1][r] = min( dpL[l][r] + (p[l] − p[l−1])·u(l, r),  dpR[l][r] + (p[r] − p[l−1])·u(l, r) )`.
- Mở sang phải (từ `[l, r]` thành `[l, r+1]`):
  `dpR[l][r+1] = min( dpR[l][r] + (p[r+1] − p[r])·u(l, r),  dpL[l][r] + (p[r+1] − p[l])·u(l, r) )`.
- Đáp án: `min(dpL[0][K−1], dpR[0][K−1])`.

**Bất biến.** `dp*[l][r]` là tổng phạt (đến thời điểm đoạn mở được tới `[l, r]`) của mọi yêu cầu nằm trong `[l, r]`, cộng phần phạt đã "đặt cọc" cho các yêu cầu ngoài `[l, r]` (đúng bằng `d × u` đã cộng dồn).

**Quy tắc nếu… thì…:** mỗi bước chọn mở sang trái hoặc sang phải; chi phí của bước là `quãng_đường × số_yêu_cầu_chưa_xong`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP bitmask (người sửa chữa), `m ≤ 15`

Không dùng cấu trúc đường thẳng: `g[mask][j]` = tổng phạt nhỏ nhất, đã phục vụ tập `mask`, thang đang ở yêu cầu `j`. Khi thêm yêu cầu `k ∉ mask`: `g[mask | k][k] = g[mask][j] + |floor_j − floor_k| × (m − |mask|)` — vì `m − |mask|` yêu cầu chưa phục vụ (gồm `k`) cùng chịu quãng đường đó. Khởi đầu `g[{j}][j] = |start − floor_j| × m`.

Nếu `start` là một yêu cầu thì yêu cầu đó phục vụ tại thời điểm 0: ta xử lý bằng cách bỏ nó khỏi tập (không tính phạt) và coi `m' = m − 1`.

In [7]:
def elevator_bitmask(n, start, requests):
    floors = [f for f in requests if f != start]      # yêu cầu tại start: phạt 0
    m = len(floors)
    if m == 0:
        return 0
    INF = float("inf")
    full = (1 << m) - 1
    g = [[INF] * m for _ in range(1 << m)]
    for j in range(m):
        g[1 << j][j] = abs(start - floors[j]) * m
    for mask in range(1, 1 << m):
        remaining = m - bin(mask).count("1")
        for j in range(m):
            cur = g[mask][j]
            if cur == INF:
                continue
            for k in range(m):
                if not (mask >> k) & 1:
                    cost = cur + abs(floors[j] - floors[k]) * remaining
                    if cost < g[mask | (1 << k)][k]:
                        g[mask | (1 << k)][k] = cost
    return min(g[full])

for n, start, reqs, e in EXAMPLES:
    print(elevator_bitmask(n, start, reqs), e)

6 6
10 10
22 22


**Độ phức tạp.** `O(m² · 2^m)` thời gian, `O(m · 2^m)` bộ nhớ. Dùng làm trọng tài cho `m ≤ 12`.

### 7.2. Cách 3 — DP đoạn `O(m²)` với bảng 2 chiều

**Giải thích từng đoạn.**

1. `p = sorted(set(requests) | {start})`, `s = index(start)`, `is_req`, `pref`, `tot`.
2. Hai bảng `dpL`, `dpR` kích thước `K × K` (chỉ dùng ô `l ≤ s ≤ r`), khởi tạo vô cực; `dpL[s][s] = dpR[s][s] = 0`.
3. Duyệt `l` từ `s` xuống `0`, `r` từ `s` lên `K−1`; từ ô `(l, r)` đẩy sang `(l−1, r)` và `(l, r+1)` theo công thức 6.3.
4. Đáp án là `min(dpL[0][K−1], dpR[0][K−1])`.

In [8]:
def elevator_interval_2d(n, start, requests):
    p = sorted(set(requests) | {start})
    K_ = len(p)
    s = p.index(start)
    req = set(requests)
    pref = [0]
    for x in p:
        pref.append(pref[-1] + (1 if x in req else 0))
    tot = pref[-1]
    INF = float("inf")
    dpL = [[INF] * K_ for _ in range(K_)]
    dpR = [[INF] * K_ for _ in range(K_)]
    dpL[s][s] = dpR[s][s] = 0
    for l in range(s, -1, -1):
        for r in range(s, K_):
            a, b = dpL[l][r], dpR[l][r]
            if a == INF and b == INF:
                continue
            u = tot - (pref[r + 1] - pref[l])
            if l > 0:
                c = min(a + (p[l] - p[l - 1]) * u, b + (p[r] - p[l - 1]) * u)
                if c < dpL[l - 1][r]:
                    dpL[l - 1][r] = c
            if r + 1 < K_:
                c = min(b + (p[r + 1] - p[r]) * u, a + (p[r + 1] - p[l]) * u)
                if c < dpR[l][r + 1]:
                    dpR[l][r + 1] = c
    return min(dpL[0][K_ - 1], dpR[0][K_ - 1])

for n, start, reqs, e in EXAMPLES:
    print(elevator_interval_2d(n, start, reqs), e)

6 6
10 10
22 22


**Độ phức tạp.** Chỉ các ô `l ≤ s ≤ r` được dùng: `(s+1) · (K−s)` ô, tối đa `~K²/4`; mỗi ô `O(1)`. Thời gian `O(m²)`, bộ nhớ `O(m²)` (hai bảng `K×K`). Với `m = 1500` bảng có 2.25M ô mỗi cái (≈ 36MB con trỏ) — chấp nhận được nhưng ta có bản gọn hơn.

### 7.3. Cách 4 — DP đoạn cuộn hàng, `O(m)` bộ nhớ (bản nộp)

Ô `(l, r)` chỉ phụ thuộc hàng `l+1` (cho `dpL`) và ô `(l, r−1)` cùng hàng (cho `dpR`). Vì vậy ta giữ **một hàng** `(dpL_row, dpR_row)` theo `r` và cập nhật hàng mới từ hàng cũ khi `l` giảm. Dưới dạng "kéo":

- `newL[r]` (đang ở `p[l]`, đoạn `[l, r]`) từ hàng cũ `[l+1, r]`: `min(oldL[r] + (p[l+1] − p[l])·u(l+1, r), oldR[r] + (p[r] − p[l])·u(l+1, r))`, với `u(l+1, r) = tot − (pref[r+1] − pref[l+1])`.
- `newR[r]` (đang ở `p[r]`) từ ô cùng hàng `[l, r−1]`: `min(newR[r−1] + (p[r] − p[r−1])·u(l, r−1), newL[r−1] + (p[r] − p[l])·u(l, r−1))`.

Khởi tạo hàng `l = s`: `L[s] = R[s] = 0`, và `R[r]` cho `r > s` tính từ chính nó (chỉ mở sang phải).

In [9]:
def elevator_interval_rolling(n, start, requests):
    p = sorted(set(requests) | {start})
    K_ = len(p)
    s = p.index(start)
    req = set(requests)
    pref = [0]
    for x in p:
        pref.append(pref[-1] + (1 if x in req else 0))
    tot = pref[-1]
    INF = float("inf")

    # hàng l = s: chỉ mở sang phải
    L = [INF] * K_
    R = [INF] * K_
    L[s] = R[s] = 0
    for r in range(s + 1, K_):
        u = tot - (pref[r] - pref[s])                    # đoạn [s, r-1]
        R[r] = R[r - 1] + (p[r] - p[r - 1]) * u
        # L[r] vẫn INF: không thể đứng ở p[s] với đoạn [s, r], r > s, mà chưa từng mở trái (đứng ở p[r])
    for l in range(s - 1, -1, -1):
        newL = [INF] * K_
        newR = [INF] * K_
        for r in range(s, K_):
            u_old = tot - (pref[r + 1] - pref[l + 1])    # đoạn [l+1, r]
            a, b = L[r], R[r]
            if a != INF or b != INF:
                newL[r] = min(a + (p[l + 1] - p[l]) * u_old, b + (p[r] - p[l]) * u_old)
        for r in range(s + 1, K_):
            u_prev = tot - (pref[r] - pref[l])           # đoạn [l, r-1]
            c1 = newR[r - 1] + (p[r] - p[r - 1]) * u_prev
            c2 = newL[r - 1] + (p[r] - p[l]) * u_prev
            newR[r] = min(c1, c2)
        L, R = newL, newR
    return min(L[K_ - 1], R[K_ - 1])

for n, start, reqs, e in EXAMPLES:
    print(elevator_interval_rolling(n, start, reqs), e)

6 6
10 10
22 22


**Độ phức tạp.** Thời gian `O(m²)`, bộ nhớ `O(m)`.

**Lưu ý chi tiết.** Với `l = s` (chưa mở trái), `L[r]` luôn là `INF` cho `r > s` (thang không thể đứng ở `p[s]` mà đoạn đã phủ tới `r` nếu chưa từng mở trái — vì khi chỉ mở phải thang ở `p[r]`). Khi `l < s`, `L[r]` với `r > s` hợp lệ.

## 8. TRACE từng bước

Trace Cách 3 trên **ví dụ 3**: `start = 5`, `requests = [0, 2, 9]`. Các tầng quan trọng `p = [0, 2, 5, 9]`, `s = 2`, `is_req = [1, 1, 0, 1]`, `tot = 3`, `pref = [0, 1, 2, 2, 3]`.

| Ô `(l, r)` | `dpL` (đang ở `p[l]`) | `dpR` (đang ở `p[r]`) | Ghi chú |
|---|---|---|---|
| `(2, 2)` | 0 | 0 | đang ở tầng 5, `u = 3` |
| `(1, 2)` | 9 | INF | mở trái tới tầng 2: `0 + (5−2)·3 = 9` (3 yêu cầu cùng chờ 3 giây) |
| `(2, 3)` | INF | 12 | mở phải tới tầng 9: `0 + (9−5)·3 = 12` |
| `(0, 2)` | 13 | INF | từ `(1,2)` mở trái: `9 + (2−0)·2 = 13` (còn 2 yêu cầu chờ) |
| `(1, 3)` | 26 | 23 | `dpR`: từ `(1,2)` mở phải `9 + (9−2)·2 = 23`; `dpL`: từ `(2,3)` mở trái `12 + (9−2)·2 = 26` |
| `(0, 3)` | 28 | **22** | `dpR`: từ `(0,2)` mở phải `13 + (9−0)·1 = 22` (chỉ còn yêu cầu tầng 9 chờ 9 giây); `dpL`: `min(26 + 2·1, 23 + 9·1) = 28` |

Đáp án `min(28, 22) = 22`. Ba yêu cầu: tầng 2 lúc 3, tầng 0 lúc 5, tầng 9 lúc 14 ⇒ `3 + 5 + 14 = 22`. Kiểm tra bằng tổng cộng dồn: `9 + 4 + 9 = 22` (9: cả ba chờ 3 giây; 4: hai chờ 2 giây; 9: một chờ 9 giây). Khớp!

In [10]:
def trace_elevator(n, start, requests):
    p = sorted(set(requests) | {start})
    K_ = len(p); s = p.index(start); req = set(requests)
    pref = [0]
    for x in p:
        pref.append(pref[-1] + (1 if x in req else 0))
    tot = pref[-1]
    INF = float("inf")
    print("p =", p, "| s =", s, "| tot =", tot, "| pref =", pref)
    dpL = {}; dpR = {}
    dpL[(s, s)] = dpR[(s, s)] = 0
    order = [(l, r) for l in range(s, -1, -1) for r in range(s, K_)]
    for l, r in order:
        a, b = dpL.get((l, r), INF), dpR.get((l, r), INF)
        if a == INF and b == INF:
            continue
        u = tot - (pref[r + 1] - pref[l])
        print(f"ô (l={l}, r={r}): dpL={a}, dpR={b}, u={u}")
        if l > 0:
            c = min(a + (p[l] - p[l - 1]) * u, b + (p[r] - p[l - 1]) * u)
            print(f"    mở trái tới p[{l-1}]={p[l-1]}: dpL({l-1},{r}) ≤ {c}")
            if c < dpL.get((l - 1, r), INF): dpL[(l - 1, r)] = c
        if r + 1 < K_:
            c = min(b + (p[r + 1] - p[r]) * u, a + (p[r + 1] - p[l]) * u)
            print(f"    mở phải tới p[{r+1}]={p[r+1]}: dpR({l},{r+1}) ≤ {c}")
            if c < dpR.get((l, r + 1), INF): dpR[(l, r + 1)] = c
    ans = min(dpL.get((0, K_ - 1), INF), dpR.get((0, K_ - 1), INF))
    print("Đáp án =", ans)
    return ans

trace_elevator(*EXAMPLES[2][:3])

p = [0, 2, 5, 9] | s = 2 | tot = 3 | pref = [0, 1, 2, 2, 3]
ô (l=2, r=2): dpL=0, dpR=0, u=3
    mở trái tới p[1]=2: dpL(1,2) ≤ 9
    mở phải tới p[3]=9: dpR(2,3) ≤ 12
ô (l=2, r=3): dpL=inf, dpR=12, u=2
    mở trái tới p[1]=2: dpL(1,3) ≤ 26
ô (l=1, r=2): dpL=9, dpR=inf, u=2
    mở trái tới p[0]=0: dpL(0,2) ≤ 13
    mở phải tới p[3]=9: dpR(1,3) ≤ 23
ô (l=1, r=3): dpL=26, dpR=23, u=1
    mở trái tới p[0]=0: dpL(0,3) ≤ 28
ô (l=0, r=2): dpL=13, dpR=inf, u=1
    mở phải tới p[3]=9: dpR(0,3) ≤ 22
ô (l=0, r=3): dpL=28, dpR=22, u=0
Đáp án = 22


22

### 8.2. Trace ví dụ 1 (`start = 4`, `[1, 5]`)

`p = [1, 4, 5]`, `s = 1`, `tot = 2`. Mở phải tới tầng 5: `0 + 1·2 = 2`; rồi mở trái tới tầng 1 từ tầng 5: `2 + (5−1)·1 = 6` ⇒ **6**. Mở trái trước: `3·2 = 6`, rồi sang phải `6 + (5−1)·1 = 10`. Chọn 6.

In [11]:
trace_elevator(*EXAMPLES[0][:3])

p = [1, 4, 5] | s = 1 | tot = 2 | pref = [0, 1, 1, 2]
ô (l=1, r=1): dpL=0, dpR=0, u=2
    mở trái tới p[0]=1: dpL(0,1) ≤ 6
    mở phải tới p[2]=5: dpR(1,2) ≤ 2
ô (l=1, r=2): dpL=inf, dpR=2, u=1
    mở trái tới p[0]=1: dpL(0,2) ≤ 6
ô (l=0, r=1): dpL=6, dpR=inf, u=1
    mở phải tới p[2]=5: dpR(0,2) ≤ 10
ô (l=0, r=2): dpL=6, dpR=10, u=0
Đáp án = 6


6

## 9. Code nộp LeetCode

Bản nộp dùng Cách 4 (DP đoạn cuộn hàng). Ta tận dụng `INF` bằng một số nguyên lớn thay vì `float("inf")` để giữ số nguyên thuần (tránh trộn float với int rất lớn).

In [12]:
class Solution:
    def elevatorRequests(self, n: int, start: int, requests: list[int]) -> int:
        p = sorted(set(requests) | {start})
        K = len(p)
        s = p.index(start)
        req = set(requests)
        pref = [0] * (K + 1)
        for i, x in enumerate(p):
            pref[i + 1] = pref[i] + (1 if x in req else 0)
        tot = pref[K]
        INF = 1 << 120

        L = [INF] * K          # L[r]: ở p[l], đã phủ đoạn [l, r]
        R = [INF] * K          # R[r]: ở p[r], đã phủ đoạn [l, r]
        L[s] = R[s] = 0
        for r in range(s + 1, K):                       # hàng l = s: chỉ mở sang phải
            R[r] = R[r - 1] + (p[r] - p[r - 1]) * (tot - (pref[r] - pref[s]))

        for l in range(s - 1, -1, -1):
            newL = [INF] * K
            newR = [INF] * K
            for r in range(s, K):
                u = tot - (pref[r + 1] - pref[l + 1])    # chưa phục vụ khi đoạn là [l+1, r]
                a, b = L[r], R[r]
                if a < INF or b < INF:
                    newL[r] = min(a + (p[l + 1] - p[l]) * u, b + (p[r] - p[l]) * u)
            for r in range(s + 1, K):
                u = tot - (pref[r] - pref[l])            # chưa phục vụ khi đoạn là [l, r-1]
                newR[r] = min(newR[r - 1] + (p[r] - p[r - 1]) * u,
                              newL[r - 1] + (p[r] - p[l]) * u)
            L, R = newL, newR
        return min(L[K - 1], R[K - 1])


sol = Solution()
for n, start, reqs, e in EXAMPLES:
    assert sol.elevatorRequests(n, start, reqs) == e, (n, start, reqs)
print("3 ví dụ của đề: OK")

3 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) Tập đã chạm là đoạn.** Đường đi của thang là dãy tầng liên tiếp `f_0 = start, f_1, ..., f_T` với `|f_{i+1} − f_i| = 1`. Tập `{f_0..f_t}` là một đoạn các số nguyên liên tục (bất kỳ đường đi bước ±1 nào cũng phủ mọi tầng giữa hai tầng đã chạm). Một yêu cầu được phục vụ lần đầu đúng lúc đoạn mở rộng phủ nó; vì các tầng khác nhau nên mỗi bước mở rộng sang một yêu cầu **mới** (hoặc qua tầng không phải yêu cầu — bỏ qua).

**(b) Ta chỉ cần xét mở rộng tới yêu cầu kế tiếp.** Đi sang trái đến yêu cầu gần nhất bên trái (`p[l−1]`) hoặc sang phải đến yêu cầu gần nhất bên phải (`p[r+1]`) — mọi lộ trình thật tương ứng một thứ tự mở rộng như vậy, và đi tới yêu cầu xa hơn đi qua yêu cầu gần hơn (phục vụ nó sớm hơn, không tệ hơn).

**(c) Công thức chi phí.** Tổng phạt `= Σ_yêu cầu thời_điểm = Σ_giây (số yêu cầu chưa xong ở giây đó)` (đổi thứ tự cộng: yêu cầu có thời điểm `t` đóng góp 1 cho mỗi giây `1..t`). Với quãng đi `d` (đoạn mở rộng từ đầu hiện tại tới `p[l−1]` hoặc `p[r+1]`), số yêu cầu chưa xong suốt quãng là `u(l, r)` (yêu cầu mới chỉ xong khi tới nơi) ⇒ cộng `d · u`.

**(d) DP có cấu trúc con tối ưu.** Tổng phạt của các bước sau chỉ phụ thuộc `(l, r, đang ở đầu nào)`, không phụ thuộc cách ta đã mở tới đó. Nên lấy `min` ở mỗi trạng thái là đúng.

**(e) Yêu cầu tại `start`.** `start ∈ requests` ⇒ `is_req[s] = 1`, nó đã trong đoạn `[s, s]` với phạt 0 và `u(s, s) = tot − 1`; đúng với đề.

## 11. Phân tích Time/Space

Đặt `m = len(requests)` (≤ 1500), `K = m` hoặc `m + 1`.

| Cách | Time | Space |
|---|---|---|
| 1a. Hoán vị | `O(m! · m)` | `O(m)` |
| 1b. DP lớp thời gian | `O(T · n · 2^m)` | `O(n · 2^m)` |
| 2. Bitmask | `O(m² · 2^m)` | `O(m · 2^m)` |
| 3. DP đoạn 2D | `O(m²)` | `O(m²)` |
| 4. DP đoạn cuộn hàng | `O(m²)` | `O(m)` |

Chỉ khoảng `s · (K − s) ≤ K²/4` ô được xét. Ta đo `m = 1500`.

In [13]:
import random, time
rng = random.Random(4023)
n = 10**9
floors = rng.sample(range(n), 1500)
st = rng.randint(0, n - 1)
t = time.time(); a = Solution().elevatorRequests(n, st, floors); ta = time.time() - t
t = time.time(); b = elevator_interval_2d(n, st, floors); tb = time.time() - t
print(f"cuộn hàng (Solution): {a} ({ta:.2f}s)")
print(f"bảng 2D             : {b} ({tb:.2f}s)")
assert a == b
# Trường hợp start ở một đầu (ô sử dụng nhiều nhất)
st2 = min(floors)
t = time.time(); print("start ở mép:", Solution().elevatorRequests(n, st2, floors), f"({time.time() - t:.2f}s)")

cuộn hàng (Solution): 1100180601734 (0.27s)
bảng 2D             : 1100180601734 (0.30s)
start ở mép: 739873336518 (0.00s)


## 12. Bộ test

Ví dụ đề, edge case, rồi đối chiếu: **DP lớp thời gian** (nguyên văn đề), hoán vị, bitmask, DP đoạn 2D, DP đoạn cuộn hàng và `Solution` trên input ngẫu nhiên nhỏ có seed. Các tầng được chọn **khác nhau** theo đề.

In [14]:
FAST = [elevator_permutations, elevator_bitmask, elevator_interval_2d, elevator_interval_rolling, Solution().elevatorRequests]

EDGE = [
    (1, 0, [0], 0),                      # đã đứng đúng tầng duy nhất
    (5, 2, [2], 0),
    (5, 2, [3], 1),
    (5, 0, [4], 4),
    (6, 3, [0, 5], 9),                   # 3 -> 5 (lúc 2) rồi 5 -> 0 (lúc 7): 2 + 7
    (10, 0, [1, 2, 3], 1 + 2 + 3),       # start ở mép, đi một chiều
    (10, 5, [5, 4, 6], 0 + 1 + 3),       # start là yêu cầu: 5 lúc 0; 4 lúc 1; 6 lúc 3
]
for n, s, r, e in EDGE + EXAMPLES:
    ref = elevator_layers(n, s, r)
    assert ref == e, ("trọng tài lệch kỳ vọng", n, s, r, ref, e)
    for fn in FAST:
        assert fn(n, s, r) == e, (fn.__name__, n, s, r, fn(n, s, r), e)
print("Ví dụ đề + edge case: mọi cách đúng (khớp DP lớp thời gian)")

rng = random.Random(402301)
for _ in range(500):
    n = rng.randint(1, 8)
    m = rng.randint(1, min(4, n))
    reqs = rng.sample(range(n), m)
    s = rng.randint(0, n - 1)
    ref = elevator_layers(n, s, reqs)
    for fn in FAST:
        assert fn(n, s, reqs) == ref, (fn.__name__, n, s, reqs, fn(n, s, reqs), ref)
print("500 input nhỏ: DP lớp thời gian = hoán vị = bitmask = DP đoạn")

Ví dụ đề + edge case: mọi cách đúng (khớp DP lớp thời gian)
500 input nhỏ: DP lớp thời gian = hoán vị = bitmask = DP đoạn


In [15]:
rng = random.Random(7)
for _ in range(300):
    m = rng.randint(1, 8)
    n = rng.randint(m, 40)
    reqs = rng.sample(range(n), m)
    s = rng.randint(0, n - 1)
    ref = elevator_permutations(n, s, reqs)
    for fn in FAST:
        assert fn(n, s, reqs) == ref, (fn.__name__, n, s, reqs)
print("300 input m <= 8: mọi cách khớp hoán vị")

for _ in range(100):
    m = rng.randint(1, 12)
    n = rng.randint(m, 10**9)
    reqs = rng.sample(range(n), m)
    s = rng.randint(0, n - 1)
    assert elevator_bitmask(n, s, reqs) == Solution().elevatorRequests(n, s, reqs) == elevator_interval_2d(n, s, reqs)
print("100 input m <= 12, giá trị lớn: bitmask = DP đoạn")

reqs = [1, 5]
snap = reqs[:]
Solution().elevatorRequests(6, 4, reqs)
assert reqs == snap

300 input m <= 8: mọi cách khớp hoán vị


100 input m <= 12, giá trị lớn: bitmask = DP đoạn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Tham lam "tầng gần nhất trước"

**Vì sao sai:** đi sang gần có thể làm các yêu cầu bên kia phải chờ lâu hơn. Ví dụ số lượng yêu cầu hai bên khác nhau.

In [16]:
def wrong_nearest_first(n, start, requests):
    left, pos, t, total = list(requests), start, 0, 0
    while left:
        left.sort(key=lambda f: abs(pos - f))
        f = left.pop(0)
        t += abs(pos - f); pos = f; total += t
    return total

rng = random.Random(3)
shown = 0
for _ in range(3000):
    n = rng.randint(5, 30)
    reqs = rng.sample(range(n), rng.randint(2, 5))
    s = rng.randint(0, n - 1)
    a, b = wrong_nearest_first(n, s, reqs), Solution().elevatorRequests(n, s, reqs)
    if a != b:
        print("phản ví dụ:", n, s, reqs, "| tham lam:", a, "| đúng:", b)
        shown += 1
        if shown >= 2:
            break

phản ví dụ: 21 8 [20, 19, 17, 3, 16] | tham lam: 85 | đúng: 69
phản ví dụ: 16 13 [12, 10, 15, 0, 9] | tham lam: 43 | đúng: 39


### Lỗi 2 — Tính tổng thời điểm **muộn nhất** (makespan) thay vì **tổng** thời điểm

**Vì sao sai:** đề yêu cầu tổng phạt của **mọi** yêu cầu. Bản "thời điểm xong cuối cùng" là bài khác (Elevator Requests III/IV).

### Lỗi 3 — Quên nhân quãng đường với **số yêu cầu chưa xong**

**Code sai:** `dp += quãng_đường` (chỉ cộng quãng).
**Vì sao sai:** mỗi giây có `u` yêu cầu chờ ⇒ phạt `u` mỗi giây. **Code đúng:** `dp += quãng_đường * u`.

### Lỗi 4 — Đếm `u` sai (gồm hoặc không gồm yêu cầu sắp tới)

**Code sai:** dùng `u` **sau** khi phục vụ yêu cầu sắp tới.
**Vì sao sai:** yêu cầu sắp tới vẫn đang chờ suốt quãng đường tới nó, nên phải nằm trong `u`.
**Code đúng:** `u(l, r) = tot − (số yêu cầu trong [l, r])` tính **trước** khi mở rộng.

In [17]:
def wrong_u_after(n, start, requests):
    """DP đoạn nhưng dùng u SAU khi phục vụ yêu cầu mới (thiếu một yêu cầu đang chờ)."""
    p = sorted(set(requests) | {start}); K_ = len(p); s = p.index(start); req = set(requests)
    pref = [0]
    for x in p:
        pref.append(pref[-1] + (1 if x in req else 0))
    tot = pref[-1]; INF = float("inf")
    dpL = [[INF] * K_ for _ in range(K_)]; dpR = [[INF] * K_ for _ in range(K_)]
    dpL[s][s] = dpR[s][s] = 0
    for l in range(s, -1, -1):
        for r in range(s, K_):
            a, b = dpL[l][r], dpR[l][r]
            if a == INF and b == INF:
                continue
            if l > 0:
                u = tot - (pref[r + 1] - pref[l - 1])        # SAI: tính trên đoạn đã mở xong
                dpL[l - 1][r] = min(dpL[l - 1][r], a + (p[l] - p[l - 1]) * u, b + (p[r] - p[l - 1]) * u)
            if r + 1 < K_:
                u = tot - (pref[r + 2] - pref[l])
                dpR[l][r + 1] = min(dpR[l][r + 1], b + (p[r + 1] - p[r]) * u, a + (p[r + 1] - p[l]) * u)
    return min(dpL[0][K_ - 1], dpR[0][K_ - 1])

for n, s, r, e in EXAMPLES:
    print("sai:", wrong_u_after(n, s, r), "| đúng:", e)

sai: 1 | đúng: 6
sai: 2 | đúng: 10
sai: 8 | đúng: 22


### Lỗi 5 — Không xử lý `start` thuộc `requests`

**Vì sao sai:** nếu bỏ `start` khỏi danh sách điểm (hoặc coi nó là một yêu cầu thêm), tổng sai. **Code đúng:** `p = sorted(set(requests) | {start})`, `is_req` chỉ đánh dấu các yêu cầu thật.

## 14. Mẹo & nhận dạng pattern

- **"Tổng thời điểm hoàn thành" trên đường thẳng** (traveling repairman / minimum latency) → DP đoạn `O(m²)` với chi phí `quãng × số_việc_còn_lại`.
- **Tập đã thăm là đoạn liên tục** (di chuyển liên tục trên đường thẳng) → trạng thái `(l, r, đầu nào)`.
- **Tính tổng theo "số phần tử còn lại"** là mẹo chung: biến tổng thời điểm tuyệt đối thành tổng `d × còn_lại`.
- Dùng `pref` để lấy `u(l, r)` trong `O(1)`.
- DP đoạn chỉ cần ô `l ≤ s ≤ r`, giảm hằng số 4 lần; cuộn hàng giảm bộ nhớ xuống `O(m)`.
- Dùng bitmask DP (người sửa chữa) làm trọng tài cho `m` nhỏ.

## 15. Biến thể

1. **Cực tiểu hoá thời điểm hoàn thành muộn nhất** (makespan) với `arrival` bằng 0: công thức đóng `R − L + min(s − L, R − s)` (Bài 5).
2. **Có trọng số** (mỗi yêu cầu có độ ưu tiên `w_i`): `u(l, r)` trở thành **tổng trọng số** còn lại.
3. **Có thời điểm xuất hiện** (arrival): bản IV/III (Elevator Requests III, IV).
4. **Phải về `start` sau khi xong:** cộng quãng về (nhưng tổng phạt không đổi vì chỉ tính thời điểm phục vụ).
5. **Hai thang máy:** chia yêu cầu thành hai nhóm, cực tiểu hoá tổng.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. Hoán vị | Thử mọi thứ tự | `O(m!·m)` | `O(m)` | Trọng tài, `m ≤ 8` |
| 1b. DP lớp thời gian | Mô phỏng nguyên văn | `O(T·n·2^m)` | `O(n·2^m)` | Trọng tài, `n` nhỏ |
| 2. Bitmask | `g[mask][j]` + `d × còn_lại` | `O(m²·2^m)` | `O(m·2^m)` | `m ≤ 15` |
| 3. DP đoạn 2D | Mở rộng đoạn | `O(m²)` | `O(m²)` | Học, `m` vừa |
| 4. DP đoạn cuộn hàng | Giữ một hàng | `O(m²)` | `O(m)` | **Bản nộp**, `m = 1500` |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `penalty_of_order(2, [0, 4])` bằng bao nhiêu?

**Bài 2 (tính tay).** `n = 10`, `start = 2`, `requests = [0, 4, 5]`. Thử các thứ tự, lập bảng thời điểm và cho đáp án.

**Bài 3 (điền TODO).** Hoàn thiện `unserved(pref, tot, l, r)` trả về số yêu cầu **chưa** phục vụ khi đoạn đã phủ là `[l, r]`.

**Bài 4 (tự viết lại).** Viết `my_elevator(n, start, requests)` theo DP đoạn (bảng 2D hoặc cuộn hàng).

**Bài 5 (biến thể).** Thay vì tổng, cực tiểu hoá **thời điểm phục vụ yêu cầu cuối cùng** (makespan). Viết `min_makespan(start, requests)` bằng công thức đóng và đối chiếu bằng hoán vị.

In [18]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def unserved(pref, tot, l, r):
    # TODO: số yêu cầu trong đoạn chỉ số [l, r] là pref[r+1] - pref[l]; trả về phần còn lại
    pass

# Bài 4 — TODO
def my_elevator(n, start, requests):
    # TODO: p = sorted(set(requests) | {start}); dpL, dpR; mở trái / mở phải với chi phí quãng * u
    pass

# Bài 5 — TODO
def min_makespan(start, requests):
    # TODO: L = min(requests + [start]); R = max(requests + [start]); trả về (R - L) + min(start - L, R - start)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: `2 → 0` mất 2 giây, `0 → 4` mất 4 giây.
- Mức 2: thời điểm `2` và `6`.
- Mức 3: `2 + 6 = 8`.

**Bài 2.**
- Mức 1: thử "phải trước": `4, 5, 0`: thời điểm `2, 3, 8` ⇒ 13.
- Mức 2: "trái trước": `0, 4, 5`: `2, 6, 7` ⇒ 15; `4, 0, 5`: `2, 6, 11` ⇒ 19.
- Mức 3: đáp án **13**.

**Bài 3.**
- Mức 1: dùng prefix sum.
- Mức 2: số trong đoạn `= pref[r+1] − pref[l]`.
- Mức 3: `return tot - (pref[r + 1] - pref[l])`.

**Bài 4.**
- Mức 1: xem Cách 3 (bảng 2D).
- Mức 2: duyệt `l` từ `s` xuống, `r` từ `s` lên.
- Mức 3: chi phí mỗi bước `quãng * unserved(...)` với đoạn **trước** khi mở rộng.

**Bài 5.**
- Mức 1: phải đi phủ cả đoạn `[L, R]`.
- Mức 2: bắt đầu bằng chạm đầu gần hơn rồi sang đầu kia.
- Mức 3: `(R − L) + min(start − L, R − start)` với `L, R` lấy trên `requests ∪ {start}`.

## 19. Đáp án tham khảo

In [19]:
# Bài 1, 2
assert penalty_of_order(2, [0, 4]) == 8
assert Solution().elevatorRequests(10, 2, [0, 4, 5]) == 13
assert penalty_of_order(2, [4, 5, 0]) == 2 + 3 + 8

# Bài 3
def unserved_ans(pref, tot, l, r):
    return tot - (pref[r + 1] - pref[l])
assert unserved_ans([0, 1, 2, 2, 3], 3, 1, 2) == 2
assert unserved_ans([0, 1, 2, 2, 3], 3, 0, 3) == 0
print("Bài 1–3 OK")

Bài 1–3 OK


In [20]:
# Bài 4
def my_elevator_ans(n, start, requests):
    p = sorted(set(requests) | {start})
    K_ = len(p); s = p.index(start); req = set(requests)
    pref = [0]
    for x in p:
        pref.append(pref[-1] + (1 if x in req else 0))
    tot = pref[-1]; INF = float("inf")
    dpL = [[INF] * K_ for _ in range(K_)]; dpR = [[INF] * K_ for _ in range(K_)]
    dpL[s][s] = dpR[s][s] = 0
    for l in range(s, -1, -1):
        for r in range(s, K_):
            a, b = dpL[l][r], dpR[l][r]
            if a == INF and b == INF:
                continue
            u = unserved_ans(pref, tot, l, r)
            if l > 0:
                dpL[l - 1][r] = min(dpL[l - 1][r], a + (p[l] - p[l - 1]) * u, b + (p[r] - p[l - 1]) * u)
            if r + 1 < K_:
                dpR[l][r + 1] = min(dpR[l][r + 1], b + (p[r + 1] - p[r]) * u, a + (p[r + 1] - p[l]) * u)
    return min(dpL[0][K_ - 1], dpR[0][K_ - 1])

# Bài 5
def min_makespan_ans(start, requests):
    pts = list(requests) + [start]
    L, R = min(pts), max(pts)
    return (R - L) + min(start - L, R - start)

def makespan_bruteforce(start, requests):
    best = float("inf")
    for order in permutations(requests):
        t, pos = 0, start
        for f in order:
            t += abs(pos - f); pos = f
        best = min(best, t)
    return best

rng = random.Random(99)
for _ in range(300):
    m = rng.randint(1, 6)
    n = rng.randint(m, 30)
    reqs = rng.sample(range(n), m)
    s = rng.randint(0, n - 1)
    assert my_elevator_ans(n, s, reqs) == Solution().elevatorRequests(n, s, reqs)
    assert min_makespan_ans(s, reqs) == makespan_bruteforce(s, reqs), (s, reqs)
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template DP đoạn (tổng thời điểm, đường thẳng):**

```python
p = sorted(set(requests) | {start}); s = p.index(start)
pref[i+1] = pref[i] + (p[i] là yêu cầu)
u(l, r) = tot - (pref[r+1] - pref[l])                 # số yêu cầu chưa xong, tính TRƯỚC khi mở
dpL[s][s] = dpR[s][s] = 0
mở trái : dpL[l-1][r] = min(dpL[l][r] + (p[l]-p[l-1])*u,  dpR[l][r] + (p[r]-p[l-1])*u)
mở phải : dpR[l][r+1] = min(dpR[l][r] + (p[r+1]-p[r])*u, dpL[l][r] + (p[r+1]-p[l])*u)
đáp án  = min(dpL[0][K-1], dpR[0][K-1])
```

**Checklist trước khi submit:**

- [ ] Điểm quan trọng = yêu cầu ∪ `{start}`, sắp tăng.
- [ ] Chi phí mỗi bước = `quãng × (số yêu cầu chưa xong trước khi đi)`.
- [ ] Chỉ xét ô `l ≤ s ≤ r`; khởi tạo `dp[s][s] = 0`.
- [ ] `start` thuộc `requests` được tính là yêu cầu phục vụ ở thời điểm 0.
- [ ] Số nguyên lớn: dùng `INF` là số nguyên lớn hoặc `float("inf")` nhất quán.
- [ ] Đối chiếu bitmask / DP lớp thời gian trên input nhỏ có seed.